# Chapter 5: Handling Missing Values

## Identifying missing values

In [1]:
import polars as pl

In [2]:
from datetime import date
import numpy as np

date_col = pl.date_range(date(2023, 1, 1), date(2023, 1, 15), '1d', eager=True)
avg_temp_c_list = [-3,None,6,-1,np.nan,6,4,None,1,2,np.nan,7,9,-2,None]
df = pl.DataFrame({
    'date': date_col,
    'avg_temp_celsius': avg_temp_c_list
}, strict=False)

In [3]:
print(df.head())

shape: (5, 2)
┌────────────┬──────────────────┐
│ date       ┆ avg_temp_celsius │
│ ---        ┆ ---              │
│ date       ┆ f64              │
╞════════════╪══════════════════╡
│ 2023-01-01 ┆ -3.0             │
│ 2023-01-02 ┆ null             │
│ 2023-01-03 ┆ 6.0              │
│ 2023-01-04 ┆ -1.0             │
│ 2023-01-05 ┆ NaN              │
└────────────┴──────────────────┘


### How to do it...

In [4]:
df.null_count()

date,avg_temp_celsius
u32,u32
0,3


In [5]:
df.select('avg_temp_celsius').null_count()

avg_temp_celsius
u32
3


In [6]:
df.select('date', 'avg_temp_celsius').null_count()

date,avg_temp_celsius
u32,u32
0,3


In [13]:
print(df)

shape: (15, 2)
┌────────────┬──────────────────┐
│ date       ┆ avg_temp_celsius │
│ ---        ┆ ---              │
│ date       ┆ f64              │
╞════════════╪══════════════════╡
│ 2023-01-01 ┆ -3.0             │
│ 2023-01-02 ┆ null             │
│ 2023-01-03 ┆ 6.0              │
│ 2023-01-04 ┆ -1.0             │
│ 2023-01-05 ┆ NaN              │
│ …          ┆ …                │
│ 2023-01-11 ┆ NaN              │
│ 2023-01-12 ┆ 7.0              │
│ 2023-01-13 ┆ 9.0              │
│ 2023-01-14 ┆ -2.0             │
│ 2023-01-15 ┆ null             │
└────────────┴──────────────────┘


In [7]:
df.select(pl.col('avg_temp_celsius').null_count())

In [8]:
df.select(
    pl.col('avg_temp_celsius')
    .is_null()
    .sum()
).item()

3

In [9]:
(
    df
    .filter(pl.col('avg_temp_celsius').is_null())
    .select(pl.len())
)

len
u32
3


In [11]:
df.filter(pl.col('avg_temp_celsius').is_null()).shape

(3, 2)

In [12]:
df.select(
    pl.col('avg_temp_celsius')
    .is_nan()
    .sum()
)

avg_temp_celsius
u32
2


In [15]:
(
    df
    .filter(pl.col('avg_temp_celsius').is_nan())
    .select(pl.len())
)

len
u32
2


## Deleting rows and columns containing missing values

### How to do it...

In [2]:
import polars as pl

In [16]:
df = pl.read_csv('../data/temperatures.csv') 

In [17]:
print(df)

shape: (15, 2)
┌────────────┬──────────────────┐
│ date       ┆ avg_temp_celsius │
│ ---        ┆ ---              │
│ str        ┆ f64              │
╞════════════╪══════════════════╡
│ 2023-01-01 ┆ -3.0             │
│ 2023-01-02 ┆ null             │
│ 2023-01-03 ┆ 6.0              │
│ 2023-01-04 ┆ -1.0             │
│ 2023-01-05 ┆ NaN              │
│ …          ┆ …                │
│ 2023-01-11 ┆ NaN              │
│ 2023-01-12 ┆ 7.0              │
│ 2023-01-13 ┆ 9.0              │
│ 2023-01-14 ┆ -2.0             │
│ 2023-01-15 ┆ null             │
└────────────┴──────────────────┘


In [23]:
df.drop_nulls()

date,avg_temp_celsius
str,f64
"""2023-01-01""",-3.0
"""2023-01-03""",6.0
"""2023-01-04""",-1.0
"""2023-01-05""",NaN
"""2023-01-06""",6.0
…,…
"""2023-01-10""",2.0
"""2023-01-11""",NaN
"""2023-01-12""",7.0


In [22]:
df.select(
    pl.col('avg_temp_celsius')
    .drop_nulls()
)    

avg_temp_celsius
f64
-3.0
6.0
-1.0
NaN
6.0
…
2.0
NaN
7.0


In [21]:
df.drop_nulls(subset=['avg_temp_celsius'])

date,avg_temp_celsius
str,f64
"""2023-01-01""",-3.0
"""2023-01-03""",6.0
"""2023-01-04""",-1.0
"""2023-01-05""",NaN
"""2023-01-06""",6.0
…,…
"""2023-01-10""",2.0
"""2023-01-11""",NaN
"""2023-01-12""",7.0


In [25]:
res = df.filter(pl.col('avg_temp_celsius').is_not_null())
print(res)

shape: (12, 2)
┌────────────┬──────────────────┐
│ date       ┆ avg_temp_celsius │
│ ---        ┆ ---              │
│ str        ┆ f64              │
╞════════════╪══════════════════╡
│ 2023-01-01 ┆ -3.0             │
│ 2023-01-03 ┆ 6.0              │
│ 2023-01-04 ┆ -1.0             │
│ 2023-01-05 ┆ NaN              │
│ 2023-01-06 ┆ 6.0              │
│ …          ┆ …                │
│ 2023-01-10 ┆ 2.0              │
│ 2023-01-11 ┆ NaN              │
│ 2023-01-12 ┆ 7.0              │
│ 2023-01-13 ┆ 9.0              │
│ 2023-01-14 ┆ -2.0             │
└────────────┴──────────────────┘


In [26]:
cols_to_drop = [column for column in df.columns if df.select(pl.col(column).is_null().any())[0,0]]
df.drop(cols_to_drop).columns

['date']

In [30]:
df.select(pl.col('date').is_null().any())

date
bool
false


In [31]:
cols_to_drop = [column for column in df.columns if df.select(pl.col(column).is_null().any()).item()]
df.drop(cols_to_drop).columns

['date']

In [32]:
df.select(
    pl.col('avg_temp_celsius')
    .drop_nans()
    .is_nan()
    .sum()
)

avg_temp_celsius
u32
0


In [33]:
df.filter(pl.col('avg_temp_celsius').is_not_nan())

date,avg_temp_celsius
str,f64
"""2023-01-01""",-3.0
"""2023-01-03""",6.0
"""2023-01-04""",-1.0
"""2023-01-06""",6.0
"""2023-01-07""",4.0
"""2023-01-09""",1.0
"""2023-01-10""",2.0
"""2023-01-12""",7.0
"""2023-01-13""",9.0


In [34]:
import polars.selectors as cs
cols_to_drop = df.select(cs.float().is_nan().any()).columns
df.drop(cols_to_drop).columns

['date']

In [35]:
df.select(pl.exclude(cols_to_drop)).columns

['date']

In [38]:
df.select(cs.float())

avg_temp_celsius
f64
-3.0
null
6.0
-1.0
NaN
…
NaN
7.0
9.0


### There is more...

In [39]:
df.fill_nan(None).drop_nulls()

date,avg_temp_celsius
str,f64
"""2023-01-01""",-3.0
"""2023-01-03""",6.0
"""2023-01-04""",-1.0
"""2023-01-06""",6.0
"""2023-01-07""",4.0
"""2023-01-09""",1.0
"""2023-01-10""",2.0
"""2023-01-12""",7.0
"""2023-01-13""",9.0


## Filling missing values

### How to do it...

In [40]:
df = pl.read_csv('../data/temperatures.csv') 
df.head()

date,avg_temp_celsius
str,f64
"""2023-01-01""",-3.0
"""2023-01-02""",null
"""2023-01-03""",6.0
"""2023-01-04""",-1.0
"""2023-01-05""",NaN


In [8]:
df.select(
    'avg_temp_celsius',
    avg_temp_nulls_filled=pl.col('avg_temp_celsius').fill_null(pl.lit('1'))
)

avg_temp_celsius,avg_temp_nulls_filled
f64,str
-3.0,"""-3.0"""
null,"""1"""
6.0,"""6.0"""
-1.0,"""-1.0"""
NaN,"""NaN"""
6.0,"""6.0"""
4.0,"""4.0"""
null,"""1"""
1.0,"""1.0"""


In [10]:
df.select(
    'avg_temp_celsius',
    forward_filled=pl.col('avg_temp_celsius').fill_null(strategy='forward'),
    backward_filled=pl.col('avg_temp_celsius').fill_null(strategy='backward'),
    mean_filled=pl.col('avg_temp_celsius').fill_null(strategy='mean'),
    min_filled=pl.col('avg_temp_celsius').fill_null(strategy='min'),
    max_filled=pl.col('avg_temp_celsius').fill_null(strategy='max'),
)

avg_temp_celsius,forward_filled,backward_filled,mean_filled,min_filled,max_filled
f64,f64,f64,f64,f64,f64
-3.0,-3.0,-3.0,-3.0,-3.0,-3.0
null,-3.0,6.0,NaN,-3.0,9.0
6.0,6.0,6.0,6.0,6.0,6.0
-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
NaN,NaN,NaN,NaN,NaN,NaN
6.0,6.0,6.0,6.0,6.0,6.0
4.0,4.0,4.0,4.0,4.0,4.0
null,4.0,1.0,NaN,-3.0,9.0
1.0,1.0,1.0,1.0,1.0,1.0


In [41]:
df.select(
    'avg_temp_celsius',
    interpolated_linear=pl.col('avg_temp_celsius').interpolate(),
    interpolated_nearest=pl.col('avg_temp_celsius').interpolate(method='nearest')
)

avg_temp_celsius,interpolated_linear,interpolated_nearest
f64,f64,f64
-3.0,-3.0,-3.0
null,1.5,6.0
6.0,6.0,6.0
-1.0,-1.0,-1.0
NaN,NaN,NaN
…,…,…
NaN,NaN,NaN
7.0,7.0,7.0
9.0,9.0,9.0


In [27]:
df.select(
    'avg_temp_celsius',
    avg_temp_median=pl.col('avg_temp_celsius')
        .fill_null(
            pl.col('avg_temp_celsius').median()
        ),
    avg_temp_max_minus_min=pl.col('avg_temp_celsius')
        .fill_null(
            pl.col('avg_temp_celsius').max() - pl.col('avg_temp_celsius').min()
        )
)

avg_temp_celsius,avg_temp_median,avg_temp_max_minus_min
f64,f64,f64
-3.0,-3.0,-3.0
null,5.0,12.0
6.0,6.0,6.0
-1.0,-1.0,-1.0
NaN,NaN,NaN
6.0,6.0,6.0
4.0,4.0,4.0
null,5.0,12.0
1.0,1.0,1.0


In [42]:
import numpy as np
import scipy
import plotly.express as px

In [43]:
x = np.linspace(0, 10, num=11)
y = np.cos(-x**2 / 9.0)
xnew = np.linspace(0, 10, num=1001)
ynew = np.interp(xnew, x, y)

In [49]:
fig = px.line(
    x=xnew,
    y=ynew,
    labels={"x": "x", "y": "y"},
    title="原始数据与线性插值",
    template="plotly_white",
)
fig.update_traces(name="线性插值", showlegend=True)

points = px.scatter(x=x, y=y)
points.update_traces(
    name="原始数据",
    showlegend=True,
    marker=dict(color="#EF553B", size=9),
)

fig.add_traces(points.data)
fig.show()

### There is more...

In [50]:
df = pl.DataFrame(
    {'values': [1,2,None,None,None,3,4,None,5]}
)
df

values
i64
1
2
null
null
null
3
4
null
5


In [51]:
df.select(
    'values',
    forward_fill=pl.col('values').forward_fill(),
    forward_fill_1=pl.col('values').forward_fill(limit=1),
    backward_fill=pl.col('values').backward_fill(),
    backward_fill_2=pl.col('values').backward_fill(limit=2),
)

values,forward_fill,forward_fill_1,backward_fill,backward_fill_2
i64,i64,i64,i64,i64
1,1,1,1,1
2,2,2,2,2
null,2,2,3,null
null,2,null,3,3
null,2,null,3,3
3,3,3,3,3
4,4,4,4,4
null,4,4,5,5
5,5,5,5,5
